<a href="https://colab.research.google.com/github/sheinafathurr1/mlp/blob/main/data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pymupdf requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 23.4 MB/s eta 0:00:00


In [ ]:
from google.colab import files

uploaded = files.upload()  # choose one or many PDF files
print("Uploaded files:", list(uploaded.keys()))

Saving document.pdf to document.pdf
Saving document (1).pdf to document (1).pdf
Uploaded files: ['document.pdf', 'document (1).pdf']


In [ ]:
import re
import csv
import fitz
import os
import pandas as pd
from io import StringIO

# =====================================================
# Helper functions
# =====================================================
def clean_text(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def count_words(text):
    return len(re.findall(r"\b[\w-]+\b", text))

def count_sentences(text):
    s = re.split(r"(?<=[.!?])\s+", text.strip())
    s = [x for x in s if x.strip()]
    return len(s)

def tokenize(text):
    return re.findall(r"\b[a-zA-Z][a-zA-Z0-9-]*\b", text.lower())

def lexical_diversity(text):
    words = tokenize(text)
    return round(len(set(words)) / len(words), 6) if words else 0.0

def avg_sentence_length(text):
    s = re.split(r"(?<=[.!?])\s+", text.strip())
    lengths = [count_words(x) for x in s if count_words(x) > 0]
    return round(sum(lengths) / len(lengths), 6) if lengths else 0.0

# =====================================================
# PDF parsing
# =====================================================
def extract_pages_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        pages.append(clean_text(page.get_text("text")))
    doc.close()
    full_text = clean_text("\n\n".join(pages))
    return pages, full_text

def detect_title(pdf_path):
    doc = fitz.open(pdf_path)
    page = doc[0]
    blocks = page.get_text("dict")["blocks"]
    doc.close()

    candidates = []
    for block in blocks:
        if "lines" not in block:
            continue

        block_text = []
        max_size = 0
        y0 = block.get("bbox", [0, 0, 0, 0])[1]

        for line in block["lines"]:
            spans = line.get("spans", [])
            line_text = "".join(span.get("text", "") for span in spans).strip()
            if line_text:
                block_text.append(line_text)
            for span in spans:
                max_size = max(max_size, span.get("size", 0))

        text = clean_text(" ".join(block_text))
        if len(text) >= 10 and len(text.split()) <= 40:
            candidates.append((y0, max_size, text))

    if not candidates:
        return ""

    stop_words = ["abstract", "introduction", "keywords", "index terms", "received"]
    filtered = []
    for y0, size, text in candidates:
        low = text.lower()
        if any(low.startswith(w) for w in stop_words):
            continue
        filtered.append((y0, size, text))

    if not filtered:
        filtered = candidates

    filtered.sort(key=lambda x: (-x[1], x[0]))
    return filtered[0][2]

def extract_abstract(full_text):
    m = re.search(
        r"\babstract\b\s*[:.\-]?\s*(.*?)\s*(?=\b(?:keywords|index terms|introduction|1\.?\s+introduction)\b)",
        full_text,
        flags=re.IGNORECASE | re.DOTALL
    )
    if m:
        return clean_text(m.group(1))

    m = re.search(r"\babstract\b\s*[:.\-]?\s*", full_text, flags=re.IGNORECASE)
    if m:
        tail = full_text[m.end():]
        return clean_text(" ".join(tail.split()[:250]))

    return ""

def extract_keywords(full_text):
    m = re.search(
        r"\b(?:keywords|index terms)\b\s*[:\-]?\s*(.*?)(?=\n|\b(?:introduction|1\.?\s+introduction)\b)",
        full_text,
        flags=re.IGNORECASE | re.DOTALL
    )
    if not m:
        return []
    raw = clean_text(m.group(1))
    parts = re.split(r"[;,•·]", raw)
    return [p.strip(" .") for p in parts if p.strip()]

def count_sections(full_text):
    patterns = [
        r"(?m)^(?:\d+\.?\s+)[A-Z][^\n]{2,80}$",
        r"(?m)^(?:[IVXLC]+\.?\s+)[A-Z][^\n]{2,80}$",
        r"(?m)^(?:[A-Z][A-Z \-]{3,80})$",
        r"(?m)^(?:\d+\.\d+\.?\s+)[A-Z][^\n]{2,80}$",
    ]
    found = []
    for p in patterns:
        found.extend(re.findall(p, full_text))
    return len(set(x.strip().lower() for x in found))

def extract_references_text(full_text):
    m = re.search(r"\b(?:references|bibliography)\b", full_text, flags=re.IGNORECASE)
    if not m:
        return ""
    return clean_text(full_text[m.end():])

def count_reference_entries(ref_text):
    if not ref_text:
        return 0

    patterns = [
        r"(?m)^\[\d+\]",
        r"(?m)^\d+\.\s",
        r"(?m)^\d+\)\s",
    ]
    counts = [len(re.findall(p, ref_text)) for p in patterns]
    best = max(counts) if counts else 0

    if best == 0:
        best = len(re.findall(r"\b10\.\d{4,9}/[-._;()/:A-Z0-9]+\b", ref_text, flags=re.IGNORECASE))

    return best

def count_figures(text):
    return len(re.findall(r"\b(?:figure|fig\.)\s*\d+\b", text, flags=re.IGNORECASE))

def count_tables(text):
    return len(re.findall(r"\btable\s*\d+\b", text, flags=re.IGNORECASE))

def count_equations(text):
    patterns = [
        r"(?m)^\s*\(\d+\)\s*$",
        r"\beq\.?\s*\(\d+\)",
        r"\bequation\s*\(\d+\)",
    ]
    return sum(len(re.findall(p, text, flags=re.IGNORECASE)) for p in patterns)

def count_intext_citations(text):
    patterns = [
        r"\[\d+(?:\s*[,;-]\s*\d+)*\]",
        r"\([A-Z][A-Za-z]+(?:\s+et al\.)?,\s*\d{4}\)",
    ]
    return sum(len(re.findall(p, text)) for p in patterns)

def extract_year(text):
    years = re.findall(r"\b(19\d{2}|20\d{2})\b", text)
    years = [int(y) for y in years if 1900 <= int(y) <= 2099]
    if not years:
        return ""
    freq = {}
    for y in years:
        freq[y] = freq.get(y, 0) + 1
    return max(freq, key=freq.get)

# =====================================================
# One PDF -> one row
# =====================================================
def extract_row(pdf_path):
    try:
        pages, full_text = extract_pages_text(pdf_path)
        title = detect_title(pdf_path)
        abstract = extract_abstract(full_text)
        keywords = extract_keywords(full_text)
        ref_text = extract_references_text(full_text)

        row = {
            "file_name": os.path.basename(pdf_path),
            "year": extract_year(full_text),
            "number_of_pages": len(pages),
            "title_word_count": count_words(title),
            "title_char_count": len(title),
            "abstract_word_count": count_words(abstract),
            "abstract_sentence_count": count_sentences(abstract),
            "fulltext_word_count": count_words(full_text),
            "fulltext_sentence_count": count_sentences(full_text),
            "number_of_sections": count_sections(full_text),
            "number_of_references": count_reference_entries(ref_text),
            "number_of_figures": count_figures(full_text),
            "number_of_tables": count_tables(full_text),
            "number_of_equations": count_equations(full_text),
            "number_of_keywords": len(keywords),
            "intext_citation_count": count_intext_citations(full_text),
            "fulltext_lexical_diversity": lexical_diversity(full_text),
            "avg_sentence_length": avg_sentence_length(full_text),
            "status": "ok",
            "error": ""
        }
        return row

    except Exception as e:
        return {
            "file_name": os.path.basename(pdf_path),
            "year": "",
            "number_of_pages": "",
            "title_word_count": "",
            "title_char_count": "",
            "abstract_word_count": "",
            "abstract_sentence_count": "",
            "fulltext_word_count": "",
            "fulltext_sentence_count": "",
            "number_of_sections": "",
            "number_of_references": "",
            "number_of_figures": "",
            "number_of_tables": "",
            "number_of_equations": "",
            "number_of_keywords": "",
            "intext_citation_count": "",
            "fulltext_lexical_diversity": "",
            "avg_sentence_length": "",
            "status": "failed",
            "error": str(e).replace("\n", " ")
        }

# =====================================================
# Run batch extraction
# =====================================================
pdf_files = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]

rows = []
for i, pdf_file in enumerate(pdf_files, start=1):
    print(f"Processing {i}/{len(pdf_files)}: {pdf_file}")
    rows.append(extract_row(pdf_file))

df = pd.DataFrame(rows)
df.to_csv("pdf_features.csv", index=False, encoding="utf-8")

print("\nSaved file: pdf_features.csv")
df

Processing 1/2: document.pdf
Processing 2/2: document (1).pdf

Saved file: pdf_features.csv


,file_name,year,number_of_pages,title_word_count,title_char_count,abstract_word_count,abstract_sentence_count,fulltext_word_count,fulltext_sentence_count,number_of_sections,number_of_references,number_of_figures,number_of_tables,number_of_equations,number_of_keywords,intext_citation_count,fulltext_lexical_diversity,avg_sentence_length,status,error
0,document.pdf,2024,21,13,101,204,8,7752,543,35,29,27,5,7,4,68,0.224915,14.276243,ok,
1,document (1).pdf,2020,16,15,137,195,12,5537,554,42,31,13,19,5,5,76,0.230385,9.994585,ok,
